In [ ]:
import os
import torch
from torchvision import datasets, transforms
from torchvision.utils import save_image

In [ ]:
root_dir = "/home/dhruv/Documents/Dhruv_CV/VARIATIONAL-AUTOENCODER-WITH-ARBITRARY-CONDITIONING-VAEAC-/Data"
mask_types = ["horizontal", "vertical"]
folders = [
    "original",
    "horizontal_mask",
    "horizontal_mask_observed_image",
    "horizontal_mask_unobserved_image",
    "vertical_mask",
    "vertical_mask_observed_image",
    "vertical_mask_unobserved_image"
]


# Create folder structure
for f in folders:
    for label in range(10):
        os.makedirs(os.path.join(root_dir, f, str(label)), exist_ok=True)

In [ ]:
def generate_horizontal_mask(h, w, num_lines=1, center_fraction=0.4):
    """
    Generate horizontal line mask (1 = visible, 0 = masked).
    Lines are centered around the middle of the image.
    """
    mask = torch.ones((1, h, w))
    
    # Define central region for line placement
    center_start = int(h * (0.5 - center_fraction / 2))
    center_end = int(h * (0.5 + center_fraction / 2))
    
    for _ in range(num_lines):
        # Sample line only within the central vertical band
        y = torch.randint(center_start, center_end, (1,)).item()
        mask[:, y, :] = 0.0
    
    return mask


def generate_vertical_mask(h, w, num_lines=1, center_fraction=0.4):
    """
    Generate vertical line mask (1 = visible, 0 = masked).
    Lines are centered around the middle of the image.
    """
    mask = torch.ones((1, h, w))
    
    # Define central region for line placement
    center_start = int(w * (0.5 - center_fraction / 2))
    center_end = int(w * (0.5 + center_fraction / 2))
    
    for _ in range(num_lines):
        # Sample line only within the central horizontal band
        x = torch.randint(center_start, center_end, (1,)).item()
        mask[:, :, x] = 0.0
    
    return mask


In [6]:
transform = transforms.ToTensor()
mnist = datasets.MNIST(root="./", train=True, download=True, transform=transform)

for i, (img, label) in enumerate(mnist):
    c, h, w = img.shape

    # Generate line masks
    horiz_mask = generate_horizontal_mask(h, w, num_lines=1)
    vert_mask = generate_vertical_mask(h, w, num_lines=1)

    # Apply masks
    horiz_masked_obs = img * horiz_mask
    horiz_masked_unobs = img * (1 - horiz_mask)

    vert_masked_obs = img * vert_mask
    vert_masked_unobs = img * (1 - vert_mask)

    save_image(img, os.path.join(root_dir, "original", str(label), f"{i:05d}.png"))
    save_image(horiz_mask, os.path.join(root_dir, "horizontal_mask", str(label), f"{i:05d}.png"))
    save_image(horiz_masked_obs, os.path.join(root_dir, "horizontal_mask_observed_image", str(label), f"{i:05d}.png"))
    save_image(horiz_masked_unobs, os.path.join(root_dir, "horizontal_mask_unobserved_image", str(label), f"{i:05d}.png"))
    save_image(vert_mask, os.path.join(root_dir, "vertical_mask", str(label), f"{i:05d}.png"))
    save_image(vert_masked_obs, os.path.join(root_dir, "vertical_mask_observed_image", str(label), f"{i:05d}.png"))
    save_image(vert_masked_unobs, os.path.join(root_dir, "vertical_mask_unobserved_image", str(label), f"{i:05d}.png"))


    if i % 1000 == 0:
        print(f"Processed {i}/{len(mnist)} images")

print("Done! All images and masks saved in:", root_dir)


100.0%
100.0%
100.0%
100.0%


Processed 0/60000 images
Processed 1000/60000 images
Processed 2000/60000 images
Processed 3000/60000 images
Processed 4000/60000 images
Processed 5000/60000 images
Processed 6000/60000 images
Processed 7000/60000 images
Processed 8000/60000 images
Processed 9000/60000 images
Processed 10000/60000 images
Processed 11000/60000 images
Processed 12000/60000 images
Processed 13000/60000 images
Processed 14000/60000 images
Processed 15000/60000 images
Processed 16000/60000 images
Processed 17000/60000 images
Processed 18000/60000 images
Processed 19000/60000 images
Processed 20000/60000 images
Processed 21000/60000 images
Processed 22000/60000 images
Processed 23000/60000 images
Processed 24000/60000 images
Processed 25000/60000 images
Processed 26000/60000 images
Processed 27000/60000 images
Processed 28000/60000 images
Processed 29000/60000 images
Processed 30000/60000 images
Processed 31000/60000 images
Processed 32000/60000 images
Processed 33000/60000 images
Processed 34000/60000 image

In [8]:
import os
import shutil
import random
from tqdm import tqdm

In [9]:
root_dir = "/home/dhruv/Documents/Dhruv_CV/VARIATIONAL-AUTOENCODER-WITH-ARBITRARY-CONDITIONING-VAEAC-/Data"


folders = {
    "original": "original",
    "horizontal_mask": "horizontal_mask",
    "horizontal_observed": "horizontal_mask_observed_image",
    "horizontal_unobserved": "horizontal_mask_unobserved_image",
    "vertical_mask": "vertical_mask",
    "vertical_observed": "vertical_mask_observed_image",
    "vertical_unobserved": "vertical_mask_unobserved_image"
}

output_root = os.path.join(root_dir, "split_data")
train_ratio = 0.8
random.seed(42)  # for reproducibility

def make_dir(path):
    if not os.path.exists(path):
        os.makedirs(path)

# Create train/test directories for each folder type
for split in ["train", "test"]:
    for folder in folders.values():
        make_dir(os.path.join(output_root, split, folder))

# Collect all (label, filename) pairs from "original"
all_images = []
for label in range(10):
    label_dir = os.path.join(root_dir, folders["original"], str(label))
    for filename in os.listdir(label_dir):
        all_images.append((label, filename))

random.shuffle(all_images)

# Split into train/test
split_index = int(len(all_images) * train_ratio)
train_data = all_images[:split_index]
test_data = all_images[split_index:]

# Helper function to copy and rename files
def copy_data(data_list, split):
    print(f"Copying {split} data ({len(data_list)} samples)...")
    for idx, (label, filename) in enumerate(tqdm(data_list, total=len(data_list))):
        new_filename = f"{idx + 1}.png"  # rename sequentially
        for folder_name in folders.values():
            src = os.path.join(root_dir, folder_name, str(label), filename)
            dst = os.path.join(output_root, split, folder_name, new_filename)
            if os.path.exists(src):
                shutil.copy(src, dst)

# Copy data
copy_data(train_data, "train")
copy_data(test_data, "test")


Copying train data (48000 samples)...


100%|██████████| 48000/48000 [00:12<00:00, 3930.15it/s]


Copying test data (12000 samples)...


100%|██████████| 12000/12000 [00:03<00:00, 3914.44it/s]


# Fashion Mnist

In [1]:
# 0 T-shirt/top
# 1 Trouser
# 2 Pullover
# 3 Dress
# 4 Coat
# 5 Sandal
# 6 Shirt
# 7 Sneaker
# 8 Bag
# 9 Ankle boot


In [ ]:
import os
import random
import shutil
from tqdm import tqdm

import torch
from torchvision import datasets, transforms
from torchvision.utils import save_image

DATASET = "FashionMNIST"  
ROOT_DIR = "/home/dhruv/Documents/Dhruv_CV/VARIATIONAL-AUTOENCODER-WITH-ARBITRARY-CONDITIONING-VAEAC-/Data/FashionMNIST"
DOWNLOAD_ROOT = "./data"
TRAIN_RATIO = 0.8
RANDOM_SEED = 42
NUM_LINES = 1
CENTER_FRACTION = 0.4

folders = [
    "original",
    "horizontal_mask",
    "horizontal_mask_observed_image",
    "horizontal_mask_unobserved_image",
    "vertical_mask",
    "vertical_mask_observed_image",
    "vertical_mask_unobserved_image"
]

for f in folders:
    for label in range(10):
        os.makedirs(os.path.join(ROOT_DIR, f, str(label)), exist_ok=True)

def generate_horizontal_mask(h, w, num_lines=1, center_fraction=0.4):
    mask = torch.ones((1, h, w))
    center_start = int(h * (0.5 - center_fraction / 2))
    center_end = int(h * (0.5 + center_fraction / 2))
    for _ in range(num_lines):
        y = torch.randint(center_start, max(center_start + 1, center_end), (1,)).item()
        mask[:, y, :] = 0.0
    return mask

def generate_vertical_mask(h, w, num_lines=1, center_fraction=0.4):
    mask = torch.ones((1, h, w))
    center_start = int(w * (0.5 - center_fraction / 2))
    center_end = int(w * (0.5 + center_fraction / 2))
    for _ in range(num_lines):
        x = torch.randint(center_start, max(center_start + 1, center_end), (1,)).item()
        mask[:, :, x] = 0.0
    return mask

transform = transforms.ToTensor()

if DATASET.lower() == "mnist":
    dataset = datasets.MNIST(root=DOWNLOAD_ROOT, train=True, download=True, transform=transform)
elif DATASET.lower() in ["fashionmnist", "fashion_mnist", "fashion-mnist"]:
    dataset = datasets.FashionMNIST(root=DOWNLOAD_ROOT, train=True, download=True, transform=transform)
else:
    raise ValueError("Unsupported DATASET. Use 'MNIST' or 'FashionMNIST'.")

print(f"Processing {len(dataset)} images from {DATASET} ...")
for i, (img, label) in enumerate(tqdm(dataset, total=len(dataset))):
    c, h, w = img.shape

    horiz_mask = generate_horizontal_mask(h, w, num_lines=NUM_LINES, center_fraction=CENTER_FRACTION)
    vert_mask = generate_vertical_mask(h, w, num_lines=NUM_LINES, center_fraction=CENTER_FRACTION)

    horiz_masked_obs = img * horiz_mask
    horiz_masked_unobs = img * (1 - horiz_mask)

    vert_masked_obs = img * vert_mask
    vert_masked_unobs = img * (1 - vert_mask)

    fname = f"{i:05d}.png"
    save_image(img, os.path.join(ROOT_DIR, "original", str(label), fname))
    # Save mask as image (single channel)
    save_image(horiz_mask, os.path.join(ROOT_DIR, "horizontal_mask", str(label), fname))
    save_image(horiz_masked_obs, os.path.join(ROOT_DIR, "horizontal_mask_observed_image", str(label), fname))
    save_image(horiz_masked_unobs, os.path.join(ROOT_DIR, "horizontal_mask_unobserved_image", str(label), fname))
    save_image(vert_mask, os.path.join(ROOT_DIR, "vertical_mask", str(label), fname))
    save_image(vert_masked_obs, os.path.join(ROOT_DIR, "vertical_mask_observed_image", str(label), fname))
    save_image(vert_masked_unobs, os.path.join(ROOT_DIR, "vertical_mask_unobserved_image", str(label), fname))

    if (i + 1) % 10000 == 0:
        print(f"Saved {i+1} images...")

print("Finished saving images and masks.")

output_root = os.path.join(ROOT_DIR, "split_data")
folders_map = {
    "original": "original",
    "horizontal_mask": "horizontal_mask",
    "horizontal_observed": "horizontal_mask_observed_image",
    "horizontal_unobserved": "horizontal_mask_unobserved_image",
    "vertical_mask": "vertical_mask",
    "vertical_observed": "vertical_mask_observed_image",
    "vertical_unobserved": "vertical_mask_unobserved_image"
}

def make_dir(path):
    os.makedirs(path, exist_ok=True)

for split in ["train", "test"]:
    for folder_name in folders_map.values():
        make_dir(os.path.join(output_root, split, folder_name))


all_images = []
for label in range(10):
    label_dir = os.path.join(ROOT_DIR, folders_map["original"], str(label))
    filenames = sorted(os.listdir(label_dir))
    for fn in filenames:
        all_images.append((label, fn))

random.seed(RANDOM_SEED)
random.shuffle(all_images)

split_index = int(len(all_images) * TRAIN_RATIO)
train_data = all_images[:split_index]
test_data = all_images[split_index:]

def copy_data(data_list, split):
    print(f"Copying {split} data ({len(data_list)} samples)...")
    for idx, (label, filename) in enumerate(tqdm(data_list, total=len(data_list))):
        new_filename = f"{idx + 1}.png"
        for folder_name in folders_map.values():
            src = os.path.join(ROOT_DIR, folder_name, str(label), filename)
            dst = os.path.join(output_root, split, folder_name, new_filename)
            if os.path.exists(src):
                shutil.copy(src, dst)
            else:
                pass

copy_data(train_data, "train")
copy_data(test_data, "test")

print("Done! Train/test split created at:", output_root)


100%|██████████| 26.4M/26.4M [00:05<00:00, 4.79MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 204kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 3.02MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 5.67MB/s]


Processing 60000 images from FashionMNIST ...


 17%|█▋        | 10161/60000 [00:12<01:01, 805.39it/s]

Saved 10000 images...


 34%|███▎      | 20102/60000 [00:24<00:48, 830.74it/s]

...

Saved 60000 images...
Finished saving images and masks.
Copying train data (48000 samples)...


100%|██████████| 48000/48000 [00:12<00:00, 3860.09it/s]


Copying test data (12000 samples)...


100%|██████████| 12000/12000 [00:03<00:00, 3726.53it/s]

Done! Train/test split created at: /home/dhruv/Documents/Dhruv_CV/VARIATIONAL-AUTOENCODER-WITH-ARBITRARY-CONDITIONING-VAEAC-/Data/FashionMNIST/split_data


# CelebA

In [2]:
!pip install gdown

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [gdown]


In [1]:
#!/usr/bin/env python3
"""
celeba_prepare_no_labels.py

Prepare CelebA images (extracted img_align_celeba/) into folders with horizontal/vertical masks,
observed/unobserved images, and create train/test split_data with sequential filenames.
No identity/class labels are used.
"""

import os
import random
import shutil
from tqdm import tqdm

from PIL import Image
import torch
from torchvision import transforms
from torchvision.utils import save_image
import pandas as pd

# ----------------- USER CONFIG -----------------
IMG_DIR = "/home/dhruv/Documents/Dhruv_CV/VARIATIONAL-AUTOENCODER-WITH-ARBITRARY-CONDITIONING-VAEAC-/Data/celeba_raw/img_align_celeba"
ROOT_DIR = "/home/dhruv/Documents/Dhruv_CV/VARIATIONAL-AUTOENCODER-WITH-ARBITRARY-CONDITIONING-VAEAC-/Data/CelebA"
TRAIN_RATIO = 0.8
RANDOM_SEED = 42
NUM_LINES = 1
CENTER_FRACTION = 0.4
LINE_THICKNESS = 1      
IMAGE_SIZE = (128,128)
OUTPUT_EXT = ".png"


folders = [
    "original",
    "horizontal_mask",
    "horizontal_mask_observed_image",
    "horizontal_mask_unobserved_image",
    "vertical_mask",
    "vertical_mask_observed_image",
    "vertical_mask_unobserved_image"
]

def safe_makedirs(path):
    os.makedirs(path, exist_ok=True)

def generate_horizontal_mask(h, w, num_lines=1, center_fraction=0.4, thickness=1):
    """Return single-channel mask (1 = observed, 0 = unobserved) of shape (1,h,w)."""
    mask = torch.ones((1, h, w), dtype=torch.float32)
    center_start = int(h * (0.5 - center_fraction / 2))
    center_end = max(center_start + 1, int(h * (0.5 + center_fraction / 2)))
    for _ in range(num_lines):
        y = torch.randint(center_start, center_end, (1,)).item()
        y0 = max(0, y - thickness // 2)
        y1 = min(h, y0 + thickness)
        mask[:, y0:y1, :] = 0.0
    return mask

def generate_vertical_mask(h, w, num_lines=1, center_fraction=0.4, thickness=1):
    mask = torch.ones((1, h, w), dtype=torch.float32)
    center_start = int(w * (0.5 - center_fraction / 2))
    center_end = max(center_start + 1, int(w * (0.5 + center_fraction / 2)))
    for _ in range(num_lines):
        x = torch.randint(center_start, center_end, (1,)).item()
        x0 = max(0, x - thickness // 2)
        x1 = min(w, x0 + thickness)
        mask[:, :, x0:x1] = 0.0
    return mask

def pil_to_tensor(img_pil, transform):
    return transform(img_pil)

def save_all(img_tensor, split_root, fname_noext):
    """
    img_tensor: C x H x W
    split_root: directory under which the 7 folders exist
    fname_noext: filename without extension (we'll append OUTPUT_EXT)
    """
    c, h, w = img_tensor.shape
    # single-channel masks then repeat to C channels for readability/saving
    horiz_mask = generate_horizontal_mask(h, w, num_lines=NUM_LINES, center_fraction=CENTER_FRACTION, thickness=LINE_THICKNESS).repeat(c, 1, 1)
    vert_mask = generate_vertical_mask(h, w, num_lines=NUM_LINES, center_fraction=CENTER_FRACTION, thickness=LINE_THICKNESS).repeat(c, 1, 1)

    horiz_obs = img_tensor * horiz_mask
    horiz_unobs = img_tensor * (1 - horiz_mask)

    vert_obs = img_tensor * vert_mask
    vert_unobs = img_tensor * (1 - vert_mask)

    fname = fname_noext + OUTPUT_EXT
    save_image(img_tensor, os.path.join(split_root, "original", fname))
    save_image(horiz_mask, os.path.join(split_root, "horizontal_mask", fname))
    save_image(horiz_obs, os.path.join(split_root, "horizontal_mask_observed_image", fname))
    save_image(horiz_unobs, os.path.join(split_root, "horizontal_mask_unobserved_image", fname))
    save_image(vert_mask, os.path.join(split_root, "vertical_mask", fname))
    save_image(vert_obs, os.path.join(split_root, "vertical_mask_observed_image", fname))
    save_image(vert_unobs, os.path.join(split_root, "vertical_mask_unobserved_image", fname))

def main():
    assert os.path.isdir(IMG_DIR), f"IMG_DIR not found: {IMG_DIR}"
    safe_makedirs(ROOT_DIR)

    # create base folders (flat, no labels)
    for f in folders:
        safe_makedirs(os.path.join(ROOT_DIR, f))

    # transform (optional resize)
    transform_list = []
    if IMAGE_SIZE is not None:
        transform_list.append(transforms.Resize(IMAGE_SIZE))
    transform_list.append(transforms.ToTensor())
    transform = transforms.Compose(transform_list)

    # list all image files
    all_fnames = sorted([f for f in os.listdir(IMG_DIR) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
    print(f"Found {len(all_fnames)} images in {IMG_DIR}")

    # first pass: save original + masks into ROOT_DIR/<folder>/<original_filename>
    print("Saving originals and masks to:", ROOT_DIR)
    for fname in tqdm(all_fnames, desc="Saving images"):
        in_path = os.path.join(IMG_DIR, fname)
        try:
            pil = Image.open(in_path).convert("RGB")
        except Exception as e:
            print(f"Warning: cannot open {in_path}: {e}")
            continue
        img_t = pil_to_tensor(pil, transform)  # CxHxW
        base_noext = os.path.splitext(fname)[0]
        save_all(img_t, ROOT_DIR, base_noext)

    # prepare split_data folders
    output_root = os.path.join(ROOT_DIR, "split_data")
    folders_map = {
        "original": "original",
        "horizontal_mask": "horizontal_mask",
        "horizontal_observed": "horizontal_mask_observed_image",
        "horizontal_unobserved": "horizontal_mask_unobserved_image",
        "vertical_mask": "vertical_mask",
        "vertical_observed": "vertical_mask_observed_image",
        "vertical_unobserved": "vertical_mask_unobserved_image"
    }

    for split in ["train", "test"]:
        for folder_name in folders_map.values():
            safe_makedirs(os.path.join(output_root, split, folder_name))

    # build list of all saved images (no labels)
    all_images = sorted([fn for fn in os.listdir(os.path.join(ROOT_DIR, "original")) if fn.lower().endswith(('.jpg', '.jpeg', '.png'))])
    print(f"Total saved original images: {len(all_images)}")

    # shuffle and split
    random.seed(RANDOM_SEED)
    shuffled = all_images[:]
    random.shuffle(shuffled)
    split_index = int(len(shuffled) * TRAIN_RATIO)
    train_list = shuffled[:split_index]
    test_list = shuffled[split_index:]

    # helper to copy & rename sequentially
    def copy_data(file_list, split):
        print(f"Copying {split} data ({len(file_list)} samples)...")
        rows = []
        for idx, filename in enumerate(tqdm(file_list, total=len(file_list))):
            new_filename = f"{idx + 1}.png"  # sequential numeric filenames
            for folder_name in folders_map.values():
                src = os.path.join(ROOT_DIR, folder_name, os.path.splitext(filename)[0] + OUTPUT_EXT)
                dst = os.path.join(output_root, split, folder_name, new_filename)
                if os.path.exists(src):
                    shutil.copy(src, dst)
                else:
                    # safety: if original saved as .jpg (unlikely since we saved as .png), fallback
                    alt_src = os.path.join(ROOT_DIR, folder_name, filename)
                    if os.path.exists(alt_src):
                        shutil.copy(alt_src, dst)
                    else:
                        # file missing — skip quietly
                        pass
            # record row mapping
            rows.append({
                "split": split,
                "orig_filename": filename,
                "new_filename": new_filename,
                "original_path": os.path.join(output_root, split, "original", new_filename),
                "horizontal_mask": os.path.join(output_root, split, "horizontal_mask", new_filename),
                "horizontal_observed": os.path.join(output_root, split, "horizontal_mask_observed_image", new_filename),
                "horizontal_unobserved": os.path.join(output_root, split, "horizontal_mask_unobserved_image", new_filename),
                "vertical_mask": os.path.join(output_root, split, "vertical_mask", new_filename),
                "vertical_observed": os.path.join(output_root, split, "vertical_mask_observed_image", new_filename),
                "vertical_unobserved": os.path.join(output_root, split, "vertical_mask_unobserved_image", new_filename),
            })
        return rows

    train_rows = copy_data(train_list, "train")
    test_rows = copy_data(test_list, "test")

    # write mappings
    pd.DataFrame(train_rows).to_csv(os.path.join(output_root, "train_mapping.csv"), index=False)
    pd.DataFrame(test_rows).to_csv(os.path.join(output_root, "test_mapping.csv"), index=False)

    print("Done! Train/test split created at:", output_root)

if __name__ == "__main__":
    main()


Found 202599 images in /home/dhruv/Documents/Dhruv_CV/VARIATIONAL-AUTOENCODER-WITH-ARBITRARY-CONDITIONING-VAEAC-/Data/celeba_raw/img_align_celeba
Saving originals and masks to: /home/dhruv/Documents/Dhruv_CV/VARIATIONAL-AUTOENCODER-WITH-ARBITRARY-CONDITIONING-VAEAC-/Data/CelebA


Saving images: 100%|██████████| 202599/202599 [3:22:26<00:00, 16.68it/s]  


Total saved original images: 202599
Copying train data (162079 samples)...


100%|██████████| 162079/162079 [02:10<00:00, 1238.69it/s]


Copying test data (40520 samples)...


100%|██████████| 40520/40520 [00:40<00:00, 1011.08it/s]


Done! Train/test split created at: /home/dhruv/Documents/Dhruv_CV/VARIATIONAL-AUTOENCODER-WITH-ARBITRARY-CONDITIONING-VAEAC-/Data/CelebA/split_data
